# 반도체 품질 예측과 검사 우선순위
**SECOM Quality Lab · 로컬 실행 포트폴리오 v1**

## tl;dr
- 공정 측정값으로 불량을 순위화하고 **제한된 검사량에서의 검출 성능**을 평가했다.
- 개발 데이터 교차검증으로 선택한 모델은 **lgbm_7**이다. 최종 평가 결과로 모델을 재선택하지 않았다.
- 시간상 마지막 **314개 / 불량 17개**를 최종 평가에 사용했다.
- 상위 10% 정책은 올림하여 32개(10.2%)를 검사한다.
  불량 **2개 / 17개**를 발견했다(검출률 **11.8%**).
  무작위 검사의 기대 발견 수는 **1.73개**, 관측 lift는 **1.15배**다.
- 최종 Average Precision(AP)은 **0.067**다.
  AP는 계단형 PR 곡선의 가중 평균이며 사다리꼴 PR-AUC와 수치가 다를 수 있다.

**핵심 결론: v1의 미래 시점 일반화는 충분하지 않다.** 상위 10% 정책의 관측 성과는 무작위 기대와 가깝다.
불량 17개인 한 시기의 결과로 실제 수율 개선, 비용 절감, 신뢰성 수명 예측을 입증한 것은 아니다.

## Context & Methods
### 면접에서 설명할 문제
“검사 자원이 제한되어 있을 때 어떤 생산 샘플을 먼저 검사할 것인가?”
불량 검출률뿐 아니라 정상 샘플 추가 검사량, 계산 시간, 재현성도 함께 설명한다.

### Key Assumptions
- 각 행은 생산 단위의 측정값이고, 목표는 최종 pass/fail이다. -1=정상, +1=불량을 0/1로 변환했다.
- timestamp는 테스트 시점이다. 모든 변수가 검사 이전에 가용하다는 보장은 없으므로 **조기 예측**이라고 주장하지 않는다.
- lot/장비 ID가 없어 동일 lot의 상관관계를 통제할 수 없다. IID bootstrap 구간은 낙관적일 수 있다.
- 앞선 약 80%에서 5-fold × 3 seed 층화 교차검증. 고정된 후보 7개를 개발 AP로 선택한다.
  개발 구간 내부 CV는 무작위 분할이므로 미래 시점 일반화의 증거는 마지막 20% 평가에서 얻는다.
- 결측률 50% 초과·상수 변수 제거, 중앙값 대체, 결측 지시자, 표준화는 **각 학습 fold 안에서만** fit한다.
- 개발 OOF 예측으로 비용 비율별 임계값을 결정한 후 최종 평가 전에 잠근다.
  점수는 보정된 확률이 아니다. 개발 OOF 비교는 모델 선택에 사용되어 낙관성이 있으며, 최종 holdout을 우선 해석한다.
- 비교표의 다른 모델들은 보조 결과다. 최종 평가에서 성능이 높은 다른 모델로 바꾸지 않는다.
- 비용 단위는 가정이다: 정상 오경보 1, 불량 미탐 5·20·100. 고정 검사량 정책과 임계값 정책은 서로 다른 정책이다.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'results').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from quality import load_data
R=ROOT/'results'
F=R/'figures'; F.mkdir(exist_ok=True)
lock=json.loads((R/'selection_lock.json').read_text(encoding='utf-8'))
champion=lock['champion']
plt.rcParams.update({'font.family':'Malgun Gothic','axes.unicode_minus':False,
                     'figure.figsize':(10,4.6),'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'axes.titlepad':14,'axes.grid':False,'savefig.dpi':150})
BLUE='#2960A4'; GOLD='#B77A16'; GRAY='#67717D'; PINK='#AE5776'
def finish(fig,name):
    fig.tight_layout()
    fig.savefig(F/name,bbox_inches='tight',facecolor='white')
    plt.show()

## Data
입력은 `data/raw/`의 사용자 다운로드 원본 복사본이다. 원본 파일의 SHA-256은 `results/protocol.json`에 기록했다.
출처: [McCann & Johnston, UCI SECOM (2008)](https://doi.org/10.24432/C54305), CC BY 4.0.
원본 설명은 591 attributes라고 쓰지만 실제 측정 행렬은 **590개 열**이다. timestamp·label은 별도 파일이다.

In [ ]:
x,y,t=load_data()
audit=json.loads((R/'data_audit.json').read_text(encoding='utf-8'))
display(pd.DataFrame({'항목':['샘플','센서','불량','정상','결측 셀','결측률','상수/무관측 변수','중복 행'],
  '값':[len(x),x.shape[1],int(y.sum()),int((y==0).sum()),audit['missing_values'],
         f"{audit['missing_fraction']:.2%}",audit['constant_sensors'],audit['duplicate_sensor_rows']]}))
monthly=pd.read_csv(R/'monthly_quality.csv')
fig,ax=plt.subplots()
ax.bar(monthly.month,monthly.failure_rate,color=BLUE,width=.6)
for i,row in monthly.iterrows():
    ax.text(i,row.failure_rate+.002,f'{int(row.failures)}/{int(row.samples)}',ha='center')
ax.set(title='월별 불량 비율 · 각 막대 위 불량 수/샘플 수',ylabel='최종 테스트 불량 비율')
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_ylim(0,max(monthly.failure_rate)*1.3)
finish(fig,'01_monthly_failure.png')

월별 비율이 달라질 수 있으므로 무작위 분할 점수만으로 결론을 내리지 않는다.
마지막 시점의 데이터는 개발 구간과 불량 비율·센서 분포가 다를 수 있다.

In [ ]:
split=pd.read_csv(R/'split_manifest.csv')
display(split.groupby('partition').agg(samples=('y','size'),failures=('y','sum'),
                                       start=('timestamp','min'),end=('timestamp','max')))
sensors=pd.read_csv(R/'sensor_audit.csv')
fig,ax=plt.subplots()
ax.hist(sensors.missing_fraction,bins=np.linspace(0,1,21),color=BLUE,edgecolor='white',rwidth=.98)
ax.axvline(.5,color=GRAY,linestyle='--',label='학습 구간에서 결측률 >50% 제거')
ax.set(title='전체 원본 센서의 결측률 분포 · 590개 변수',xlabel='센서별 결측률',ylabel='센서 수')
ax.xaxis.set_major_formatter(PercentFormatter(1)); ax.legend(frameon=False)
finish(fig,'02_missingness.png')

위 결측률은 데이터 이해용 전체 원본 통계다. 모델의 변수 제거와 대체값 계산에는 전체 통계를 사용하지 않았다.

## Results
### 1. 개발 구간에서 모델을 고르고, 미래 구간에서 평가
개발 AP의 seed별 표준편차는 분할 민감도이며 신뢰구간이 아니다.
무작위 모델의 AP 기준은 해당 평가 구간의 불량 비율이다.

In [ ]:
cv=pd.read_csv(R/'cv_summary.csv')
holdout=pd.read_csv(R/'holdout_metrics.csv')
merged=cv[['model','mean_repeat_ap','sd_repeat_ap']].merge(
    holdout[['model','average_precision','roc_auc','selected']],on='model',how='outer')
display(merged.round(4))
fig,ax=plt.subplots(figsize=(10,5.6))
ordered=holdout.set_index('model').reindex(holdout.model.iloc[::-1])
colors=[BLUE if n==champion else '#B7C6D8' for n in ordered.index]
ax.barh(ordered.index,ordered.average_precision,color=colors)
for i,(name,row) in enumerate(ordered.iterrows()):
    ax.text(row.average_precision+.004,i,f'{row.average_precision:.3f}'+('  ← 사전 선택' if name==champion else ''),va='center')
ax.axvline(ordered.prevalence.iloc[0],color=GRAY,linestyle='--',label='최종 구간 불량 비율')
ax.set(xlabel='Average Precision · 불량=양성',title='최종 시간 holdout 성능 · 314개, 불량 17개')
ax.set_xlim(0,min(1,max(ordered.average_precision)*1.5)); ax.legend(frameon=False)
finish(fig,'03_holdout_ap.png')

### 2. 검사 물량 대비 불량 검출
무작위 기대선은 실제 실험 결과가 아니라 같은 검사량에서의 수학적 기대값이다.
검사 수는 ceil(샘플 수 × 비율)로 정하며, 동점은 결과 label을 사용하지 않는 고정 해시 순서로 처리한다.

In [ ]:
budgets=pd.read_csv(R/'inspection_budgets.csv')
selected=budgets[budgets.model==champion].sort_values('actual_fraction')
display(selected[['budget','inspected','hits','missed','precision','recall',
                  'random_expected_hits','lift','random_95_low_hits','random_95_high_hits']].round(3))
fig,ax=plt.subplots()
for name,color,marker in [(champion,BLUE,'o'),('isolation_forest',GOLD,'s')]:
    frame=budgets[budgets.model==name].sort_values('actual_fraction')
    ax.plot(np.r_[0,frame.actual_fraction],np.r_[0,frame.recall],label=name,color=color,marker=marker)
ax.plot([0,1],[0,1],color=GRAY,linestyle='--',label='무작위 검사 기대값')
ax.set(xlim=(0,1.02),ylim=(0,1.05),xlabel='검사한 샘플 비율',ylabel='전체 불량 중 발견한 비율',
       title='검사량에 따른 불량 검출률 · 선은 평가한 정책 지점을 연결')
ax.xaxis.set_major_formatter(PercentFormatter(1));ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(frameon=False);finish(fig,'04_inspection_curve.png')

상위 10% 정책의 관측 검출률은 11.8%다.
고정된 모델에 대한 행 단위 bootstrap 95% 구간은
0.0%~30.0%다.
이는 학습 불확실성이나 lot 상관관계를 포함하지 않으며, 실제 생산 현장의 성능 보장이 아니다.
표의 `random_95_*`는 무작위 비복원 검사에서 발견 수의 95% 범위(초기하분포)로 bootstrap 구간과 다르다.

### 3. 비용 비율을 바꿨을 때
임계값은 개발 OOF에서 선택했다. 평가 데이터에 맞춰 임계값을 재조정하지 않았다.
아래 비용은 **FP + 미탐 비용 배수 × FN**이다. 검사 비용과 실제 제품 손실액을 모두 반영한 경제 모델은 아니다.

In [ ]:
costs=pd.read_csv(R/'cost_scenarios.csv')
display(costs.round(4))
fig,ax=plt.subplots()
idx=np.arange(len(costs));width=.25
for offset,key,label,color in [(-1,'cost_units','개발에서 고정한 임계값',BLUE),
                               (0,'inspect_none_cost','검사 안 함',GOLD),
                               (1,'inspect_all_cost','전부 양성 판정',GRAY)]:
    ax.bar(idx+offset*width,costs[key],width,label=label,color=color)
ax.set_xticks(idx,[f'미탐 비용 {int(v)}배' for v in costs.fn_cost])
ax.set(ylabel='가정된 오류 비용 단위',title='비용 가정에 따른 최종 holdout 정책 비교')
ax.legend(frameon=False);finish(fig,'05_cost_scenarios.png')

### 4. 시간 일반화 실패의 추가 진단
**이 진단은 최종 평가를 본 후 수행했다.** v1 모델·임계값을 바꾸는 근거로 사용하지 않았다.
개발 구간만 시간 순서대로 세 번 검증하고, 개발/최종 구간의 센서 분포 차이를 기술적으로 측정했다.
KS 통계량은 분포 거리이며, 아래 기준 0.2는 탐색용이다. 인과관계나 유의성 판정이 아니다.

In [ ]:
walk=pd.read_csv(R/'development_walk_forward.csv')
display(walk.pivot(index='model',columns='fold',values='average_precision').round(3))
diagnostic=json.loads((R/'diagnostic_summary.json').read_text(encoding='utf-8'))
display(pd.DataFrame({'진단':['개발 불량 비율','최종 불량 비율','KS 거리 ≥0.2 센서 수',
                              '결측률 변화 ≥10%p 센서 수'],
                      '값':[f"{diagnostic['development_failure_rate']:.2%}",
                            f"{diagnostic['holdout_failure_rate']:.2%}",
                            diagnostic['sensors_with_ks_ge_02'],diagnostic['sensors_missing_delta_ge_10pp']]}))
fig,ax=plt.subplots()
selected_walk=walk[walk.model==champion]
ax.plot(selected_walk.fold+1,selected_walk.average_precision,color=BLUE,marker='o',label=champion)
ax.plot(selected_walk.fold+1,selected_walk.prevalence,color=GRAY,marker='s',linestyle='--',label='각 구간 불량 비율')
ax.set_xticks([1,2,3]);ax.set_ylim(bottom=0)
ax.set(xlabel='개발 구간 내 시간 검증 순서',ylabel='Average Precision',title='개발 구간에서도 시간별 성능 변동 확인')
ax.legend(frameon=False);finish(fig,'07_walk_forward.png')

분포 차이와 시간별 성능 변동은 추가 조사의 단서다. 데이터가 적거나 숨은 lot 상관관계가 있는 경우도 가능해
“드리프트가 실패를 일으켰다”라고 단정하지 않는다. 최종 구간에서 로지스틱 회귀가 더 높게 나왔더라도,
그 결과를 보고 모델을 바꾸면 별도의 새 평가 데이터가 필요하다.

### 5. 모델이 참고하는 변수
개발 구간의 별도 validation fold에서 permutation importance를 측정했다.
전체 평균의 일반적 중요도나 물리적 불량 원인을 뜻하지 않는다. 익명 센서이므로 공정 조건 개선을 제안하지 않는다.

In [ ]:
importance=pd.read_csv(R/'development_permutation_importance.csv').head(10).iloc[::-1]
fig,ax=plt.subplots()
ax.barh(importance.sensor,importance.ap_drop_mean,color=BLUE,
        xerr=importance.ap_drop_sd,error_kw={'ecolor':GRAY,'capsize':3})
ax.axvline(0,color=GRAY,linewidth=.8)
ax.set(xlabel='변수를 섞었을 때 AP 감소량',title='개발 validation fold의 변수 중요도 · 오차선은 3회 섞기 표준편차')
finish(fig,'06_sensor_importance.png')
display(importance.iloc[::-1].round(4))

## Takeaways
- 문제 정의를 ‘불량 분류 정확도’에서 ‘검사 자원 배분’으로 연결했다.
- 시간 분리, fold 내부 전처리, 개발에서 고정한 선택/임계값으로 평가 오염을 줄였다.
- 데이터가 작고 불량이 드물어 점수 한 개보다 분할 민감도와 발견 수를 함께 해석해야 한다.
- 생산 적용 전에는 최근 데이터, lot/장비 단위 검증, 변수 가용 시점 확인, 확률 보정,
  실제 비용 산정, shadow 운영이 필요하다.
- **TabPFN·TabM·VAE–SHAP는 이 v1에서 실행하지 않았다.** 최신 논문은 후속 비교/확장 계획의 근거이며 구현 성과가 아니다.

### 재현 경로
`src/train.py` → `results/` 및 `models/` → 이 노트북.
실행 환경 버전은 `results/environment.json`, 정확한 패키지 목록은 `requirements-lock.txt`에 있다.
전체 학습은 `run.ps1 -Action train`, 이 노트북의 실행·HTML 변환은 `run.ps1 -Action notebook`.
이 노트북은 저장된 실험 결과를 다시 읽어 표·그림을 재현하며 모델 재학습을 숨겨서 수행하지 않는다.

### 참고 연구
1. [비용 민감 Transformer, Cluster Computing 2026](https://doi.org/10.1007/s10586-026-06080-2): 비용을 고려한 평가 설계.
2. [TabPFN, Nature 2025](https://doi.org/10.1038/s41586-024-08328-6): 후속 최신 모델 비교.
3. [MOFS-SPE, APEM 2025](https://doi.org/10.14743/apem2025.4.552): 변수 선택 확장.
4. [VAE–SHAP, Complex & Intelligent Systems 2026](https://doi.org/10.1007/s40747-026-02323-2): 합성 증강/설명 연구, 인과 추론과 구분.
5. [TabM, ICLR 2025](https://proceedings.iclr.cc/paper_files/paper/2025/hash/c1ba41c694834aeef91ae161711d4939-Abstract-Conference.html): 신경망 비교 확장.